# 03. Demonstração do modelo do grupo (desenvolvimento por CV interna)

Antes de executar este notebook, implemente `build_group_model` e `search_space` em `src/models/group_model.py` (a tabela no início do arquivo indica o ponto de entrada de cada modelo tema) e instale o extra do modelo (ver README). O espaço de busca e o wrapper devem ser justificados a priori (artigo original e documentação), não ajustados a partir de resultados.

O notebook avalia o modelo com a configuração padrão por CV interna, executa uma busca curta de hiperparâmetros e mostra um estudo de sensibilidade com `cv_grid`, tudo no treino do fold externo `OUTER_FOLD` (0, como no enunciado; o estudo pode ser repetido no treino dos demais folds externos, nunca no teste).

**Teste selado.** Este notebook pertence à fase de desenvolvimento: usa somente o TREINO de um fold externo oficial (`get_train_indices`) e mede o desempenho por validação cruzada INTERNA. O teste dos folds externos só é usado uma vez, em `python -m src.pipeline.final_test`, depois do congelamento (`python -m src.pipeline.freeze`).

**Validação é diagnóstico.** O treino de um fold externo contém o teste dos outros dois folds externos. Por isso, os resultados desta fase (`results/validation.csv`, estudos e curvas) não são base de decisões: os sistemas, o controle `group_tuned`, o wrapper, a configuração padrão, o espaço de busca e o orçamento são fixados a priori. Depois de `run_validation`, só são permitidas correções de falha de execução (status `error`, contrato do estimador), registradas no relatório; mudanças motivadas por desempenho não são permitidas.

**Ordem.** Antes de executar este notebook, comite `src/models/group_model.py` (configuração padrão e espaço de busca) e `docs/declaracoes_a_priori.md` (subconjunto de datasets dos estudos, grades e sentido de simplicidade de cada hiperparâmetro). Este notebook é demonstrativo, só deve ser executado depois desse commit e não deve motivar mudanças no espaço de busca.

In [1]:
import sys
from pathlib import Path

REPO = Path('..').resolve()
sys.path.insert(0, str(REPO))

import pandas as pd

from data.load_tabarena import get_train_indices, load_task, read_selection
from src.models.group_model import build_group_model, search_space
from src.pipeline.cv_grid import cv_grid, inner_cv_splits, stratified_subsample
from src.pipeline.evaluate import fit_predict_evaluate
from src.pipeline.tune import tune

OUTER_FOLD = 0   # fold externo cujo TREINO é usado nos estudos (enunciado: fold externo 0)

In [2]:
STUDY_TASKS = {
    'small': 363621,   # blood-transfusion-service-center
    'medium': 363704,  # students_dropout_and_academic_success
    'large': 363628,   # customer_satisfaction_in_airline
}
assert set(STUDY_TASKS.values()).issubset(read_selection())

# Demonstração inicial no representante pequeno; estudos usam o subconjunto declarado.
task_id = STUDY_TASKS['small']
ds = load_task(task_id)
train_idx = get_train_indices(task_id, folds=[OUTER_FOLD])[0]   # somente o treino do fold externo
X_train, y_train = ds.X.iloc[train_idx], ds.y[train_idx]
X_dev, y_dev = stratified_subsample(X_train, y_train, 10_000, seed=42)
splits = inner_cv_splits(y_dev, n_folds=3, seed=42)

## Configuração padrão (CV interna)

In [3]:
rows = []
for inner, (tr, va) in enumerate(splits):
    r = fit_predict_evaluate(build_group_model(seed=42), X_dev.iloc[tr], y_dev[tr],
                             X_dev.iloc[va], y_dev[va])
    rows.append({'inner_fold': inner, **r.to_dict()})
pd.DataFrame(rows)[['inner_fold', 'auc_ovo', 'accuracy', 'cross_entropy', 'total_time_s']]

,inner_fold,auc_ovo,accuracy,cross_entropy,total_time_s
0,0,0.708359,0.771084,0.510037,6.069652
1,1,0.757521,0.777108,0.462247,3.030934
2,2,0.797718,0.807229,0.444055,1.923688


## Busca de hiperparâmetros no treino (orçamento reduzido para demonstração)

In [4]:
tuning = tune(build_group_model, search_space, X_train, y_train, seed=42, n_trials=5, timeout_s=300)
print('Melhores hiperparâmetros:', tuning.best_params)
print(f'AUC OvO na CV interna (otimista: usada na seleção): {tuning.best_score:.4f}')
print(f'Tempo de busca: {tuning.tune_time_s:.1f}s')

Melhores hiperparâmetros: {'norm_methods': ['power'], 'outlier_threshold': 2.7276805754484785, 'softmax_temperature': 1.4180889157778485, 'average_logits': True}
AUC OvO na CV interna (otimista: usada na seleção): 0.7560
Tempo de busca: 27.4s


In [5]:
# Histórico da busca (útil para o relatório: convergência e sensibilidade).
tuning.study.trials_dataframe().sort_values('value', ascending=False).head(10)

,number,value,datetime_start,datetime_complete,duration,params_average_logits,params_norm_methods,params_outlier_threshold,params_softmax_temperature,state
3,3,0.755974,2026-11-06 13:03:26.426684,2026-11-06 13:03:31.584513,0 days 00:00:05.157829,True,power,2.727681,1.418089,COMPLETE
2,2,0.754597,2026-11-06 13:03:21.449384,2026-11-06 13:03:26.426684,0 days 00:00:04.977300,True,none+power+quantile_rtdl,3.782266,0.825222,COMPLETE
0,0,0.754533,2026-11-06 13:03:09.876475,2026-11-06 13:03:15.757992,0 days 00:00:05.881517,True,none+power+robust,3.045981,0.532946,COMPLETE
4,4,0.754333,2026-11-06 13:03:31.584513,2026-11-06 13:03:37.322611,0 days 00:00:05.738098,True,none+power+quantile_rtdl,4.233119,0.519251,COMPLETE
1,1,0.754193,2026-11-06 13:03:15.757992,2026-11-06 13:03:21.449384,0 days 00:00:05.691392,False,none+power+quantile_rtdl,3.136387,0.611612,COMPLETE


## Estudos de sensibilidade e estudo específico do TabICL v2 com `cv_grid`

Conforme declarado a priori em `docs/declaracoes_a_priori.md` (Seções 4 e 5), os estudos de sensibilidade (2 hiperparâmetros) e o estudo específico do modelo tema são executados por validação cruzada interna (`cv_folds=3`, `seed=42`) exclusivamente no **treino do fold externo `OUTER_FOLD = 0`** dos 3 datasets de `STUDY_TASKS` (`small`: `363621`, `medium`: `363704`, `large`: `363628`).

`cv_grid` prepara a subamostra estratificada (até `10_000` instâncias) e as partições internas (as mesmas da busca) e delega o laço a `validation_curve` (`src/pipeline/overfitting.py`), medindo treino e validação em cada fold interno. Os estudos são estritamente diagnósticos: não alteram o espaço de busca nem o `protocolo.lock`.

In [6]:
# Carrega o treino do fold externo 0 dos 3 datasets declarados em STUDY_TASKS
study_data = {}
for regime, t_id in STUDY_TASKS.items():
    task_ds = load_task(t_id)
    tr_idx = get_train_indices(t_id, folds=[OUTER_FOLD])[0]
    study_data[regime] = (
        t_id,
        task_ds.name,
        task_ds.X.iloc[tr_idx],
        task_ds.y[tr_idx],
    )

SUMMARY_COLS = ['auc_ovo_train', 'auc_ovo_val', 'cross_entropy_train', 'cross_entropy_val']


def run_study_across_regimes(param: str, values: list) -> pd.DataFrame:
    """Executa cv_grid para (param, values) nos 3 regimes de STUDY_TASKS e exibe o resumo."""
    all_grids = []
    for regime, (t_id, ds_name, X_tr, y_tr) in study_data.items():
        print(f'=== [{regime.upper()}] {t_id} - {ds_name} | {param} ===')
        grid = cv_grid(build_group_model, param, values, X_tr, y_tr, cv_folds=3, seed=42)
        summary = grid.groupby('value', sort=False)[SUMMARY_COLS].agg(['mean', 'std'])
        display(summary)
        all_grids.append(grid.assign(regime=regime, task_id=t_id, dataset=ds_name))
    return pd.concat(all_grids, ignore_index=True)

### 1. Estudo de sensibilidade 1: `softmax_temperature`

- **Grade declarada:** `[0.5, 0.7, 0.9, 1.2, 1.5]` (padrão = `0.9`)
- **Sentido de simplicidade:** maior é mais simples (`SIMPLER_IS_LOWER = False`: maior temperatura suaviza a distribuição predita e reduz a nitidez extrema).

In [7]:
PARAM_SENS_1 = 'softmax_temperature'
VALUES_SENS_1 = [0.5, 0.7, 0.9, 1.2, 1.5]

grid_softmax_temperature = run_study_across_regimes(PARAM_SENS_1, VALUES_SENS_1)

=== [SMALL] 363621 - blood-transfusion-service-center | softmax_temperature ===


auc_ovo_train           auc_ovo_val           cross_entropy_train  \
               mean       std        mean       std                mean   
value                                                                     
0.5        0.815724  0.020258    0.754533  0.044755            0.463572   
0.7        0.815724  0.020258    0.754533  0.044755            0.426121   
0.9        0.815724  0.020258    0.754533  0.044755            0.422385   
1.2        0.815724  0.020258    0.754533  0.044755            0.436698   
1.5        0.815724  0.020258    0.754533  0.044755            0.457005   

                cross_entropy_val            
            std              mean       std  
value                                        
0.5    0.027005          0.553864  0.091429  
0.7    0.020573          0.490376  0.054157  
0.9    0.017940          0.472113  0.034079  
1.2    0.016357          0.473689  0.017763  
1.5    0.015495          0.486371  0.009161

=== [MEDIUM] 363704 - students_dropout_and_academic_success | softmax_temperature ===


auc_ovo_train           auc_ovo_val           cross_entropy_train  \
               mean       std        mean       std                mean   
value                                                                     
0.5        0.990136  0.001232    0.884275  0.002546            0.168708   
0.7        0.990152  0.001223    0.884333  0.002462            0.182205   
0.9        0.990086  0.001177    0.884220  0.002264            0.211499   
1.2        0.989874  0.001155    0.883969  0.001947            0.267477   
1.5        0.989540  0.001102    0.883468  0.001692            0.326874   

                cross_entropy_val            
            std              mean       std  
value                                        
0.5    0.006308          0.667171  0.035672  
0.7    0.003124          0.571693  0.023409  
0.9    0.002186          0.542476  0.015703  
1.2    0.002483          0.546711  0.009457  
1.5    0.003531          0.572571  0.007760

=== [LARGE] 363628 - customer_satisfaction_in_airline | softmax_temperature ===


auc_ovo_train           auc_ovo_val           cross_entropy_train  \
               mean       std        mean       std                mean   
value                                                                     
0.5        0.999983  0.000014    0.991410  0.001604            0.009314   
0.7        0.999983  0.000014    0.991409  0.001605            0.014504   
0.9        0.999983  0.000014    0.991410  0.001604            0.020795   
1.2        0.999983  0.000014    0.991409  0.001605            0.031754   
1.5        0.999983  0.000014    0.991410  0.001605            0.044384   

                cross_entropy_val            
            std              mean       std  
value                                        
0.5    0.002936          0.146445  0.026694  
0.7    0.003716          0.124938  0.017356  
0.9    0.004438          0.117859  0.011876  
1.2    0.005355          0.118972  0.006623  
1.5    0.006149          0.127269  0.003002

### 2. Estudo de sensibilidade 2: `outlier_threshold`

- **Grade declarada:** `[2.5, 3.0, 4.0, 5.0, 6.0]` (padrão = `4.0`)
- **Sentido de simplicidade:** menor é mais simples (`SIMPLER_IS_LOWER = True`: menor limiar restringe mais fortemente valores extremos antes do *soft log-clipping*).

In [8]:
PARAM_SENS_2 = 'outlier_threshold'
VALUES_SENS_2 = [2.5, 3.0, 4.0, 5.0, 6.0]

grid_outlier_threshold = run_study_across_regimes(PARAM_SENS_2, VALUES_SENS_2)

=== [SMALL] 363621 - blood-transfusion-service-center | outlier_threshold ===


auc_ovo_train           auc_ovo_val           cross_entropy_train  \
               mean       std        mean       std                mean   
value                                                                     
2.5        0.813843  0.020105    0.753057  0.046262            0.423506   
3.0        0.815117  0.020630    0.753261  0.045662            0.422819   
4.0        0.815724  0.020258    0.754533  0.044755            0.422385   
5.0        0.815958  0.020343    0.754333  0.044446            0.422417   
6.0        0.816075  0.020566    0.754799  0.044629            0.422363   

                cross_entropy_val            
            std              mean       std  
value                                        
2.5    0.018003          0.471907  0.034725  
3.0    0.018080          0.471928  0.034344  
4.0    0.017940          0.472113  0.034079  
5.0    0.018010          0.472240  0.034196  
6.0    0.018094          0.472288  0.034219

=== [MEDIUM] 363704 - students_dropout_and_academic_success | outlier_threshold ===


auc_ovo_train           auc_ovo_val           cross_entropy_train  \
               mean       std        mean       std                mean   
value                                                                     
2.5        0.990176  0.001956    0.884081  0.002170            0.213669   
3.0        0.990429  0.001467    0.884226  0.002395            0.209740   
4.0        0.990086  0.001177    0.884220  0.002264            0.211499   
5.0        0.989994  0.001047    0.884292  0.002296            0.212159   
6.0        0.989842  0.000999    0.884317  0.002298            0.213301   

                cross_entropy_val            
            std              mean       std  
value                                        
2.5    0.006768          0.543317  0.015131  
3.0    0.004019          0.542590  0.016263  
4.0    0.002186          0.542476  0.015703  
5.0    0.001650          0.542462  0.015757  
6.0    0.001989          0.542327  0.015668

=== [LARGE] 363628 - customer_satisfaction_in_airline | outlier_threshold ===


auc_ovo_train           auc_ovo_val           cross_entropy_train  \
               mean       std        mean       std                mean   
value                                                                     
2.5        0.999982  0.000015    0.991428  0.001593            0.020895   
3.0        0.999982  0.000015    0.991425  0.001599            0.020860   
4.0        0.999983  0.000014    0.991410  0.001604            0.020795   
5.0        0.999983  0.000014    0.991403  0.001605            0.020769   
6.0        0.999983  0.000014    0.991402  0.001600            0.020790   

                cross_entropy_val            
            std              mean       std  
value                                        
2.5    0.004436          0.117776  0.011759  
3.0    0.004469          0.117803  0.011803  
4.0    0.004438          0.117859  0.011876  
5.0    0.004422          0.117917  0.011872  
6.0    0.004464          0.117908  0.011848

### 3. Estudo específico do modelo tema (TabICL v2)

Avalia os mecanismos próprios do conjunto de visões (*ensemble views*) do `TabICLClassifier` declarados em `docs/declaracoes_a_priori.md`:
1. **Transformações de normalização (`norm_methods`):** `['none', 'power', 'quantile_rtdl', 'robust', 'none+power']` (padrão = `'none+power'`).
2. **Estratégia de permutação de atributos (`feat_shuffle_method`):** `['none', 'shift', 'random', 'latin']` (padrão = `'latin'`).
3. **Número de estimadores no ensemble (`n_estimators`):** `[1, 2, 4, 8, 16, 32]` (padrão = `8`).

In [9]:
PARAM_SPEC_NORM = 'norm_methods'
VALUES_SPEC_NORM = ['none', 'power', 'quantile_rtdl', 'robust', 'none+power']

grid_norm_methods = run_study_across_regimes(PARAM_SPEC_NORM, VALUES_SPEC_NORM)

=== [SMALL] 363621 - blood-transfusion-service-center | norm_methods ===


auc_ovo_train           auc_ovo_val            \
                       mean       std        mean       std   
value                                                         
none               0.820820  0.020724    0.754681  0.043173   
power              0.805934  0.017525    0.755974  0.048356   
quantile_rtdl      0.826519  0.026857    0.750846  0.042597   
robust             0.822031  0.025098    0.755762  0.041739   
none+power         0.815724  0.020258    0.754533  0.044755   

              cross_entropy_train           cross_entropy_val            
                             mean       std              mean       std  
value                                                                    
none                     0.418633  0.017567          0.471351  0.032743  
power                    0.429090  0.015773          0.472413  0.035524  
quantile_rtdl            0.414945  0.023517          0.475072  0.030493  
robust                   0.417600  0.020228          0.470279  0.031180  
none+power               0.422385  0.017940          0.472113  0.034079

=== [MEDIUM] 363704 - students_dropout_and_academic_success | norm_methods ===


auc_ovo_train           auc_ovo_val            \
                       mean       std        mean       std   
value                                                         
none               0.991370  0.001505    0.884799  0.002270   
power              0.988999  0.000769    0.884544  0.002930   
quantile_rtdl      0.991636  0.003104    0.883110  0.004169   
robust             0.991366  0.003282    0.880150  0.004515   
none+power         0.990086  0.001177    0.884220  0.002264   

              cross_entropy_train           cross_entropy_val            
                             mean       std              mean       std  
value                                                                    
none                     0.202995  0.004614          0.541227  0.015134  
power                    0.219370  0.001369          0.541640  0.016802  
quantile_rtdl            0.201706  0.016709          0.545917  0.018204  
robust                   0.209515  0.016199          0.548962  0.014964  
none+power               0.211499  0.002186          0.542476  0.015703

=== [LARGE] 363628 - customer_satisfaction_in_airline | norm_methods ===


auc_ovo_train           auc_ovo_val            \
                       mean       std        mean       std   
value                                                         
none               0.999978  0.000018    0.991471  0.001660   
power              0.999988  0.000011    0.991274  0.001703   
quantile_rtdl      0.999986  0.000011    0.991409  0.001709   
robust             0.999966  0.000027    0.991016  0.001893   
none+power         0.999983  0.000014    0.991410  0.001604   

              cross_entropy_train           cross_entropy_val            
                             mean       std              mean       std  
value                                                                    
none                     0.021748  0.004957          0.117358  0.011900  
power                    0.019616  0.004441          0.118825  0.012340  
quantile_rtdl            0.020547  0.004385          0.117691  0.012084  
robust                   0.021255  0.004981          0.121734  0.014541  
none+power               0.020795  0.004438          0.117859  0.011876

In [10]:
PARAM_SPEC_SHUFFLE = 'feat_shuffle_method'
VALUES_SPEC_SHUFFLE = ['none', 'shift', 'random', 'latin']

grid_feat_shuffle = run_study_across_regimes(PARAM_SPEC_SHUFFLE, VALUES_SPEC_SHUFFLE)

=== [SMALL] 363621 - blood-transfusion-service-center | feat_shuffle_method ===


auc_ovo_train           auc_ovo_val           cross_entropy_train  \
                mean       std        mean       std                mean   
value                                                                      
none        0.816056  0.020710    0.754799  0.045184            0.422206   
shift       0.815503  0.020547    0.754336  0.044406            0.422537   
random      0.814456  0.018822    0.753402  0.044138            0.423227   
latin       0.815724  0.020258    0.754533  0.044755            0.422385   

                 cross_entropy_val            
             std              mean       std  
value                                         
none    0.018125          0.472287  0.033594  
shift   0.018008          0.472248  0.033633  
random  0.016217          0.471925  0.033987  
latin   0.017940          0.472113  0.034079

=== [MEDIUM] 363704 - students_dropout_and_academic_success | feat_shuffle_method ===


auc_ovo_train           auc_ovo_val           cross_entropy_train  \
                mean       std        mean       std                mean   
value                                                                      
none        0.991548  0.002842    0.882962  0.001591            0.201376   
shift       0.991716  0.002461    0.883016  0.001569            0.201269   
random      0.990702  0.003690    0.884010  0.001994            0.205004   
latin       0.990086  0.001177    0.884220  0.002264            0.211499   

                 cross_entropy_val            
             std              mean       std  
value                                         
none    0.012910          0.544984  0.014704  
shift   0.009792          0.545588  0.015460  
random  0.022833          0.542247  0.015045  
latin   0.002186          0.542476  0.015703

=== [LARGE] 363628 - customer_satisfaction_in_airline | feat_shuffle_method ===


auc_ovo_train           auc_ovo_val           cross_entropy_train  \
                mean       std        mean       std                mean   
value                                                                      
none        0.999989  0.000011    0.991001  0.001701            0.019069   
shift       0.999988  0.000012    0.990994  0.001708            0.019084   
random      0.999991  0.000007    0.991462  0.001749            0.016989   
latin       0.999983  0.000014    0.991410  0.001604            0.020795   

                 cross_entropy_val            
             std              mean       std  
value                                         
none    0.005516          0.120766  0.012223  
shift   0.005634          0.120816  0.012279  
random  0.004124          0.117455  0.012978  
latin   0.004438          0.117859  0.011876

In [11]:
PARAM_SPEC_ESTIMATORS = 'n_estimators'
VALUES_SPEC_ESTIMATORS = [1, 2, 4, 8, 16, 32]

grid_n_estimators = run_study_across_regimes(PARAM_SPEC_ESTIMATORS, VALUES_SPEC_ESTIMATORS)

=== [SMALL] 363621 - blood-transfusion-service-center | n_estimators ===


auc_ovo_train           auc_ovo_val           cross_entropy_train  \
               mean       std        mean       std                mean   
value                                                                     
1          0.815516  0.017327    0.752478  0.042161            0.422675   
2          0.821137  0.023080    0.752208  0.042237            0.418754   
4          0.816229  0.020715    0.754063  0.045173            0.421937   
8          0.815724  0.020258    0.754533  0.044755            0.422385   
16         0.813749  0.019036    0.755202  0.044646            0.423622   
32         0.813749  0.019036    0.755202  0.044646            0.423622   

                cross_entropy_val            
            std              mean       std  
value                                        
1      0.014906          0.470984  0.032603  
2      0.020732          0.473882  0.033182  
4      0.017964          0.472020  0.034399  
8      0.017940          0.472113  0.034079  
16     0.016464          0.471636  0.034107  
32     0.016464          0.471636  0.034107

=== [MEDIUM] 363704 - students_dropout_and_academic_success | n_estimators ===


auc_ovo_train           auc_ovo_val           cross_entropy_train  \
               mean       std        mean       std                mean   
value                                                                     
1          0.991407  0.003936    0.882950  0.000732            0.200058   
2          0.989547  0.002093    0.882558  0.001863            0.213592   
4          0.990367  0.002335    0.883692  0.002216            0.209685   
8          0.990086  0.001177    0.884220  0.002264            0.211499   
16         0.990299  0.001179    0.884855  0.002574            0.210794   
32         0.990720  0.001291    0.884471  0.002089            0.207856   

                cross_entropy_val            
            std              mean       std  
value                                        
1      0.024597          0.545329  0.009135  
2      0.006359          0.547415  0.014247  
4      0.011039          0.543991  0.015187  
8      0.002186          0.542476  0.015703  
16     0.001725          0.541049  0.015965  
32     0.001984          0.541498  0.014776

=== [LARGE] 363628 - customer_satisfaction_in_airline | n_estimators ===


auc_ovo_train           auc_ovo_val           cross_entropy_train  \
               mean       std        mean       std                mean   
value                                                                     
1          0.999988  0.000008    0.990587  0.001581            0.018445   
2          0.999982  0.000015    0.991077  0.001751            0.020705   
4          0.999979  0.000021    0.991297  0.001700            0.022258   
8          0.999983  0.000014    0.991410  0.001604            0.020795   
16         0.999984  0.000014    0.991400  0.001696            0.020621   
32         0.999988  0.000009    0.991475  0.001720            0.019542   

                cross_entropy_val            
            std              mean       std  
value                                        
1      0.004407          0.123963  0.011690  
2      0.004501          0.120385  0.013228  
4      0.005314          0.118652  0.012391  
8      0.004438          0.117859  0.011876  
16     0.004686          0.117886  0.012129  
32     0.004162          0.117448  0.012482